<a href="https://colab.research.google.com/github/erik-777/CompetenciaModelosTradicionales/blob/main/Competencia_Kaggle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Upload your kaggle.json file (contains API key)
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"erikvergara","key":"cf77a392d3b935328fa3c76f6c8b168d"}'}

In [2]:
# Make directory and move kaggle.json
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/

# Set permissions
!chmod 600 ~/.kaggle/kaggle.json

!kaggle competitions download -c aa-ii-2025-i-modelos-tradicionales-ca-mama

!unzip aa-ii-2025-i-modelos-tradicionales-ca-mama.zip

  0% 0.00/311k [00:00<?, ?B/s]
100% 311k/311k [00:00<00:00, 663MB/s]
Archive:  aa-ii-2025-i-modelos-tradicionales-ca-mama.zip
  inflating: Diccionario.xlsx        
  inflating: df_test.parquet         
  inflating: df_train.parquet        
  inflating: sample_submission.csv   


# Librerias

In [ ]:
!pip install -U imbalanced-learn

In [ ]:
# import numpy as np  # Algebra lineal, manipulación de arreglos numéricos.
# import pandas as pd  # Procesamiento de datos, lectura/escritura de archivos CSV.
# import os.path as osp  # Manejo de rutas de archivos.
# import pickle  # Serialización y deserialización de objetos Python (guardar/cargar modelos).


# # Modelos de clasificación Adicional

# from sklearn.svm import SVC



# from imblearn.over_sampling import SMOTE
# from imblearn.model_selection import RandomizedSearchCV
# # Preprocesamiento de datos
# from sklearn.preprocessing import OneHotEncoder  # Codificación one-hot para variables categóricas nominales.
# from sklearn.preprocessing import OrdinalEncoder  # Codificación ordinal para variables categóricas con orden.
# from sklearn.preprocessing import StandardScaler  # Normalización de datos para mejorar el rendimiento del modelo.
# from sklearn.preprocessing import FunctionTransformer  # Aplicación de transformaciones personalizadas.

# # División del conjunto de datos
# from sklearn.model_selection import train_test_split, StratifiedKFold  # División en conjunto de entrenamiento y prueba.

# # Selección de características
# from sklearn.feature_selection import VarianceThreshold  # Elimina características con varianza baja (irrelevantes).
# from sklearn.feature_selection import SelectPercentile, chi2  # Selección de características más relevantes con Chi-cuadrado.

# # Construcción del pipeline de procesamiento y modelado
# from sklearn.compose import ColumnTransformer  # Aplica transformaciones específicas a diferentes columnas.
# from sklearn.pipeline import Pipeline, make_pipeline  # Automatiza el flujo de preprocesamiento y modelado.

# # Manejo de valores faltantes
# from sklearn.impute import SimpleImputer  # Rellena valores faltantes con media, mediana, moda, etc.

# # Evaluación de modelos
# import sklearn.metrics as skm  # Métricas de rendimiento como precisión, recall, F1-score, AUC-ROC, etc.

# # Visualización de datos
# import matplotlib.pyplot as plt  # Gráficos y visualización de métricas.
# import seaborn as sns  # Visualización avanzada con gráficos estadísticos.

# # Medición de tiempos de ejecución
# from time import time  # Captura de tiempo de inicio y fin de ejecución.
# from datetime import timedelta  # Cálculo de diferencias de tiempo en ejecución.

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# #Funciones Utiles
# def get_imbalaced_metrics(y_true, y_preds):
#     '''calcula métricas de evaluación para modelos de clasificación cuando los datos están desbalanceados.'''
#     ths = np.linspace(0, 1, 1000)
#     best_th = ths[
#         np.argmax([skm.f1_score(y_true, y_preds>th) for th in ths])
#     ]

#     roc_auc = skm.roc_auc_score(y_true, y_preds)
#     average_precision = skm.average_precision_score(y_true, y_preds)
#     max_f1 = skm.f1_score(y_true, y_preds>best_th)
#     accuracy_on_max_f1 = skm.accuracy_score(y_true, y_preds>best_th)
#     kappa = skm.cohen_kappa_score(y_true, y_preds>best_th)
#     baseline=y_true.value_counts(True)


#     return dict(
#         roc_auc=roc_auc,
#         average_precision=average_precision,
#         max_f1=max_f1,
#         accuracy_on_max_f1=accuracy_on_max_f1,
#         kappa=kappa,
#         baseline=baseline.iloc[0],
#         best_th = best_th
#     )
# #Carga Data Set
# df = pd.read_parquet("df_train.parquet")
# df.head()

# # Divicion de Dataset
# X, y = df.drop(columns="Target"), df["Target"]
# y.value_counts(True) * 100

# # Calculamos la edad de los pacientes al momento de la complicación o corte del analisis.
# X['EDAD_COMPLICACION'] = (X['Fecha_cero'] - X['FECHA_NACIMIENTO']).dt.days // 365

# #Porcentaje de Nulidad
# porcetaje_de_nulidad = (
#     X.isnull()
#     .apply(lambda s: s.value_counts(True)).T
# )

# porcetaje_de_nulidad.columns = ['not_null', 'null']
# variables_muy_nulas = porcetaje_de_nulidad.query('null > 0.7').index

# #Convercion de tipos de Datos
# columnas_numerico=['MULTI_CANCER','RIESGOS']
# X[columnas_numerico] = X[columnas_numerico].astype(float)

# columnas_categ= ['GENERO','ESTADO_CIVIL',
#                  'CESION','CANCER_MAMA_FAMILIAR',
#                 'CANCER_OTRO_SITIO','CANCER_OTRO_SITIO_FAMILIAR','CEREBRAL_FAMILIAR'
#                 ,'atencion_nutricion'
#                 ]
# X[columnas_categ] = X[columnas_categ].astype(str)

# #Dividimos el conjunto de datos en entrenamiento y prueba, por ahora, sin implementar un protocolo complejo de evaluación.
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

# #Validemos que tan desbalanceados quedaron los particionamientos
# print(y_train.value_counts(True)*100)
# print(y_test.value_counts(True)*100)


# #Selecciona las columnas categóricas (variables tipo object o cadenas de texto) en X_train.
# categoricas = X_train.select_dtypes('object').columns
# categoricas = categoricas.delete(
#     categoricas.isin(variables_muy_nulas)
# )

# ##Selecciona las columnas numéricas en X_train (variables tipo int o float).
# numericas = X_train.select_dtypes('number').columns
# numericas = numericas.delete(
#     numericas.isin(variables_muy_nulas)
# )

# #OneHotEncoder
# config_onehot = dict(
#     handle_unknown='ignore' # Ignora cualquier categoría desconocida que aparezca en los datos de prueba pero que no estaba en los datos de entrenamiento.
# )

# #
# numeric_transformer = Pipeline(
#     steps=[("imputer",  SimpleImputer(strategy='mean')),
#            ("scaler", StandardScaler()),
#            ("select_var", VarianceThreshold(0.1))
#            ]
# )

# categorical_transformer = Pipeline(
#     steps=[('imputer', SimpleImputer(strategy='most_frequent')),
#            ('dumm', OneHotEncoder(**config_onehot)),
#            ("selector", SelectPercentile(chi2, percentile=50))
#            ]
# )

# preprocessor = ColumnTransformer(
#     transformers=[
#         ("num", numeric_transformer, numericas),
#         ("cat", categorical_transformer, categoricas),
#     ]
# )

# numeric_transformer = Pipeline(
#     steps=[("imputer",  SimpleImputer(strategy='mean')),
#            ("select_var", VarianceThreshold(0.1))
#            ]
# )

# categorical_transformer = Pipeline(
#     steps=[('imputer', SimpleImputer(strategy='most_frequent')),
#            ('dumm', OneHotEncoder(**config_onehot)),
#            ]
# )

# tree_preprocessing = ColumnTransformer(
#     transformers=[
#         ("num", numeric_transformer, numericas),
#         ("cat", categorical_transformer, categoricas),
#     ]
# )

# config_xgb = {
#     'objective': 'binary:logistic',
#     'eval_metric': 'auc',
#     'use_label_encoder': False,
#     'scale_pos_weight': 9,  # ~11% de casos positivos => ~9:1 ratio
#     'random_state': 42,
#     'tree_method': 'hist',  # Más eficiente para grandes datasets
#     'verbosity': 1
# }
# param_dist = {
#     'model__n_estimators': [100, 200, 300, 500, 700, 1000],
#     'model__max_depth': [3, 4, 5, 6, 7, 8, 9],
#     'model__learning_rate': [0.01, 0.03, 0.05, 0.07, 0.1, 0.15],
#     'model__min_child_weight': [1, 3, 5, 7],
#     'model__subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
#     'model__colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
#     'model__gamma': [0, 0.1, 0.2, 0.3, 0.5],
#     'model__reg_alpha': [0, 0.001, 0.01, 0.1, 1],
#     'model__reg_lambda': [0, 0.001, 0.01, 0.1, 1],
#     'sampling__k_neighbors': [3, 5, 7],  # Optimizamos también los parámetros de SMOTE
# }


# XGboostPipeline = Pipeline([
#     ('preprocesamiento', tree_preprocessing),
#     ('sampling', SMOTE(random_state=42)),
#     ('model', XGBClassifier(**config_xgb))
# ])

# # Validación cruzada estratificada
# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# # Búsqueda aleatoria de hiperparámetros
# print("Iniciando búsqueda de hiperparámetros...")
# start = time()
# random_search = RandomizedSearchCV(
#     estimator=XGboostPipeline,
#     param_distributions=param_dist,
#     n_iter=30,  # Puedes ajustar según tiempo disponible
#     scoring='f1',
#     n_jobs=-1,
#     cv=cv,
#     verbose=2,
#     random_state=42
# )

# random_search.fit(X_train, y_train)
# end = time()
# print(f"Tiempo de optimización: {timedelta(seconds=end-start)}")
# print("Mejores hiperparámetros encontrados:")
# print(random_search.best_params_)


# # Entrenamiento XGBoost


# # best model
# best_model = xgb_pipeline.best_estimator_

# # Reentrenar con TODOS los datos (X e y completos)
# best_model.fit(X, y)

# #Metricas
# xgb_val_preds = best_model.predict_proba(X_test)[:, 1]
# xgb_metrics = get_imbalaced_metrics(y_test, xgb_val_preds)
# xgb_metrics


# Intento 9 - Puntuacion 0.48

In [ ]:


# Importación de librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from time import time
from datetime import timedelta
import pickle
import os

import sklearn.metrics  as skm

# Modelos
from xgboost import XGBClassifier

# Preprocesamiento y evaluación
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectPercentile, VarianceThreshold, chi2
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold


# Manejo de datos desbalanceados
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE, BorderlineSMOTE

# Función para calcular métricas en datos desbalanceados
def get_imbalaced_metrics(y_true, y_preds):
    '''Calcula métricas de evaluación para modelos de clasificación cuando los datos están desbalanceados.'''
    ths = np.linspace(0, 1, 1000)
    best_th = ths[
        np.argmax([skm.f1_score(y_true, y_preds>th) for th in ths])
    ]

    roc_auc = skm.roc_auc_score(y_true, y_preds)
    average_precision = skm.average_precision_score(y_true, y_preds)
    max_f1 = skm.f1_score(y_true, y_preds>best_th)
    accuracy_on_max_f1 = skm.accuracy_score(y_true, y_preds>best_th)
    kappa = skm.cohen_kappa_score(y_true, y_preds>best_th)
    baseline=y_true.value_counts(True)

    return dict(
        roc_auc=roc_auc,
        average_precision=average_precision,
        max_f1=max_f1,
        accuracy_on_max_f1=accuracy_on_max_f1,
        kappa=kappa,
        baseline=baseline.iloc[0],
        best_th = best_th
    )

# 1. CARGA Y PREPARACIÓN DE DATOS
print("Cargando datos...")
df = pd.read_parquet("df_train.parquet")
df_test = pd.read_parquet("df_test.parquet")

# Calculamos la edad al momento de complicación
df['EDAD_COMPLICACION'] = (df['Fecha_cero'] - df['FECHA_NACIMIENTO']).dt.days // 365
df_test['EDAD_COMPLICACION'] = (df_test['Fecha_cero'] - df_test['FECHA_NACIMIENTO']).dt.days // 365

# División de datos
X = df.drop(columns=["Target"])
y = df["Target"]

# Verificamos la distribución de clases
class_distribution = y.value_counts(normalize=True) * 100
print(f"Distribución de clases: {class_distribution.iloc[1]:.2f}% positivos, {class_distribution.iloc[0]:.2f}% negativos")

# Porcentaje de nulidad
porcetaje_de_nulidad = (
    X.isnull()
    .apply(lambda s: s.value_counts(True)).T
)

porcetaje_de_nulidad.columns = ['not_null', 'null']
variables_muy_nulas = porcetaje_de_nulidad.query('null > 0.7').index

# Conversión de tipos de datos
columnas_numerico=['MULTI_CANCER','RIESGOS']
X[columnas_numerico] = X[columnas_numerico].astype(float)

#df_test[columnas_numerico] = df_test[columnas_numerico].astype(float)

columnas_categ= ['GENERO','ESTADO_CIVIL',
                 'CESION','CANCER_MAMA_FAMILIAR',
                'CANCER_OTRO_SITIO','CANCER_OTRO_SITIO_FAMILIAR','CEREBRAL_FAMILIAR',
                'atencion_nutricion'
                ]
X[columnas_categ] = X[columnas_categ].astype(str)

#df_test[columnas_categ] = df_test[columnas_categ].astype(str)

# Dividimos en entrenamiento y validación
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

# Validamos distribución en particiones
print("Distribución en conjunto de entrenamiento:")
print(y_train.value_counts(normalize=True) * 100)
print("Distribución en conjunto de validación:")
print(y_test.value_counts(normalize=True) * 100)

# 2. INGENIERÍA DE CARACTERÍSTICAS AVANZADA
def create_medical_features(X):
    """Crea características específicas para complicaciones de cáncer de mama"""
    X_new = X.copy()

    # 1. Factor de riesgo compuesto (ponderado según importancia médica)
    if 'RIESGOS' in X.columns and 'CANCER_MAMA_FAMILIAR' in X.columns:
        X_new['RIESGO_COMPUESTO'] = X['RIESGOS'].astype(float) * (1 + 0.5*X['CANCER_MAMA_FAMILIAR'].astype(float))

        # Si existe la columna de cáncer en otro sitio, la incluimos
        if 'CANCER_OTRO_SITIO' in X.columns:
            X_new['RIESGO_COMPUESTO'] += 0.3*X['CANCER_OTRO_SITIO'].astype(float)

    # 2. Edad como factor de riesgo (relación no lineal con riesgo)
    if 'EDAD_COMPLICACION' in X.columns:
        # Relación cuadrática con la edad (mayor riesgo en edades extremas)
        X_new['EDAD_CUADRATICO'] = X['EDAD_COMPLICACION']**2 / 100

        # Categorización de edad (factor médicamente relevante)
        X_new['GRUPO_EDAD'] = pd.cut(
            X['EDAD_COMPLICACION'],
            bins=[0, 40, 50, 60, 70, 100],
            labels=['<40', '40-50', '50-60', '60-70', '>70']
        )

        # Interacción edad-riesgo
        if 'RIESGOS' in X.columns:
            X_new['EDAD_RIESGO'] = X['EDAD_COMPLICACION'] * X['RIESGOS'].astype(float) / 100

    # 3. Indicador de múltiples factores de riesgo
    risk_columns = ['CANCER_MAMA_FAMILIAR', 'CANCER_OTRO_SITIO', 'CANCER_OTRO_SITIO_FAMILIAR']
    risk_columns = [col for col in risk_columns if col in X.columns]

    if len(risk_columns) > 0:
        X_new['MULT_FACTORES_RIESGO'] = X[risk_columns].astype(float).sum(axis=1)

    # 4. Variables específicas para cáncer de mama
    if 'MULTI_CANCER' in X.columns:
        # Interacción con otros factores
        if 'RIESGOS' in X.columns:
            X_new['MULTI_RIESGOS'] = X['MULTI_CANCER'].astype(float) * X['RIESGOS'].astype(float)

    # 5. Interacción de variables categóricas con numéricas
    # Edad por género
    if 'EDAD_COMPLICACION' in X.columns and 'GENERO' in X.columns:
        for genero in X['GENERO'].unique():
            col_name = f"EDAD_GENERO_{genero}"
            X_new[col_name] = X['EDAD_COMPLICACION'] * (X['GENERO'] == genero).astype(float)

    # 6. Características específicas del dominio médico para cáncer de mama
    # Riesgo por edad categórica (importante médicamente)
    if 'RIESGOS' in X.columns and 'GRUPO_EDAD' in X_new.columns:
        for grupo in X_new['GRUPO_EDAD'].cat.categories:
            col_name = f"RIESGO_EDAD_{grupo}"
            X_new[col_name] = X['RIESGOS'].astype(float) * (X_new['GRUPO_EDAD'] == grupo).astype(float)

    # 7. Interacciones de factores de riesgo específicos
    if 'CANCER_MAMA_FAMILIAR' in X.columns and 'CANCER_OTRO_SITIO' in X.columns:
        X_new['RIESGO_FAMILIAR_COMBINADO'] = X['CANCER_MAMA_FAMILIAR'].astype(float) * X['CANCER_OTRO_SITIO'].astype(float) * 2

    return X_new

# Aplicamos la ingeniería de características
print("Aplicando ingeniería de características avanzada...")
X_train_enhanced = create_medical_features(X_train)
X_test_enhanced = create_medical_features(X_test)

# 3. PREPROCESAMIENTO
print("Configurando preprocesamiento...")

# Seleccionamos columnas categóricas y numéricas (excluyendo variables muy nulas)
categoricas = X_train_enhanced.select_dtypes(['object', 'category']).columns
categoricas = categoricas[~categoricas.isin(variables_muy_nulas)]

numericas = X_train_enhanced.select_dtypes(['int64', 'float64']).columns
numericas = numericas[~numericas.isin(variables_muy_nulas)]

# Transformadores para árbol (decisión, XGBoost)
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy='mean')),
    ("select_var", VarianceThreshold(0.1))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('dumm', OneHotEncoder(handle_unknown='ignore'))
])

# Preprocesador columnar para árboles
tree_preprocessing = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numericas),
    ("cat", categorical_transformer, categoricas)
])

# 4. CONFIGURACIÓN DE XGBOOST OPTIMIZADO
print("Configurando XGBoost optimizado...")

# XGBoost optimizado para datos médicos desbalanceados
config_xgb = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'use_label_encoder': False,
    'scale_pos_weight': 9,  # ~11% de casos positivos => ~9:1 ratio
    'random_state': 42,
    'tree_method': 'hist',  # Más eficiente para grandes datasets
    'verbosity': 1
}

# 5. BÚSQUEDA DE HIPERPARÁMETROS
print("Iniciando búsqueda de hiperparámetros...")

# Espacio de búsqueda de hiperparámetros para XGBoost
param_dist = {
    'model__n_estimators': [100, 200, 300, 500, 700],
    'model__max_depth': [3, 4, 5, 6, 7, 8],
    'model__learning_rate': [0.01, 0.03, 0.05, 0.07, 0.1],
    'model__min_child_weight': [1, 3, 5, 7],
    'model__subsample': [0.7, 0.8, 0.9, 1.0],
    'model__colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'model__gamma': [0, 0.1, 0.2, 0.3],
    'model__reg_alpha': [0, 0.01, 0.1],
    'model__reg_lambda': [0, 0.01, 0.1, 1],
    'sampling__k_neighbors': [3, 5, 7]
}

# Pipeline completo: preprocesamiento + SMOTE + XGBoost
xgb_pipeline = ImbPipeline([
    ('preprocesamiento', tree_preprocessing),
    ('sampling', SMOTE(random_state=42)),
    ('model', XGBClassifier(**config_xgb))
])

# Validación cruzada estratificada
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Búsqueda aleatoria de hiperparámetros
start = time()
random_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_dist,
    n_iter=30,  # 30 iteraciones
    scoring='f1',
    cv=cv,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train_enhanced, y_train)
end = time()

print(f"Tiempo de búsqueda: {timedelta(seconds=end-start)}")
print('Mejores hiperparámetros:', random_search.best_params_)
print('Mejor F1-Score (CV):', random_search.best_score_)

# 6. EVALUACIÓN DEL MODELO OPTIMIZADO
print("\nEvaluando el modelo XGBoost optimizado...")

# Obtenemos el mejor modelo
best_model = random_search.best_estimator_

# Predicciones en validación
xgb_val_preds = best_model.predict_proba(X_test_enhanced)[:, 1]
xgb_metrics = get_imbalaced_metrics(y_test, xgb_val_preds)
print("\nMétricas del modelo optimizado:")
for key, value in xgb_metrics.items():
    print(f"{key}: {value}")

# Guardamos el umbral óptimo
best_threshold = xgb_metrics['best_th']

# 7. ENTRENAMIENTO DEL MODELO FINAL CON TODOS LOS DATOS
print("\nEntrenando modelo final con todos los datos...")
# Combinamos todo para el entrenamiento final

X_full = pd.concat([X_train, X_test])
X_full_enhanced = create_medical_features(X_full)
y_full = pd.concat([y_train, y_test])


# Entrenamiento con todos los datos
start = time()
final_pipeline = ImbPipeline([
    ('preprocesamiento', tree_preprocessing),
    ('sampling', SMOTE(random_state=42, k_neighbors=5)),
    ('model', XGBClassifier(**{k.replace('model__', ''): v
                              for k, v in random_search.best_params_.items()
                              if k.startswith('model__')}))
])

final_pipeline.fit(X_full_enhanced, y_full)
end = time()
print(f"Tiempo de entrenamiento final: {timedelta(seconds=end-start)}")


with open("modelo_xgboost_final.pkl", "wb") as f:
    pickle.dump(final_pipeline, f)

# 9. GENERAR SUBMISSION EXACTAMENTE COMO LO NECESITAS
test_df = pd.read_parquet("df_test.parquet")
test_df['EDAD_COMPLICACION'] = (test_df['Fecha_cero'] - test_df['FECHA_NACIMIENTO']).dt.days // 365

test_df[columnas_numerico] = test_df[columnas_numerico].astype(float)
test_df[columnas_categ] = test_df[columnas_categ].astype(str)

test_df_enhanced = create_medical_features(test_df)

# Predicciones exactamente como lo pediste
submission_pred = final_pipeline.predict_proba(test_df_enhanced)[:, 1]
submission_pred_bool = submission_pred > best_threshold
submission_pred_int = [int(item) for item in submission_pred_bool]
submission = pd.DataFrame(data=dict(ID=test_df.index, Target=submission_pred_int))
submission.to_csv("submission_9.csv", index=False)



# Intento 10 -

In [ ]:
# IMPLEMENTACIÓN OPTIMIZADA DE XGBOOST PARA PREDICCIÓN DE CÁNCER DE MAMA
# =====================================================================

# Importación de librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from time import time
from datetime import timedelta
import pickle
import os

import sklearn.metrics  as skm

# Modelos
from xgboost import XGBClassifier

# Preprocesamiento y evaluación
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectPercentile, VarianceThreshold, chi2
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold


# Manejo de datos desbalanceados
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE, BorderlineSMOTE, ADASYN
from imblearn.combine import SMOTEENN

# Función para calcular métricas en datos desbalanceados
def get_imbalaced_metrics(y_true, y_preds):
    '''Calcula métricas de evaluación para modelos de clasificación cuando los datos están desbalanceados.'''
    ths = np.linspace(0, 1, 1000)
    best_th = ths[
        np.argmax([skm.f1_score(y_true, y_preds>th) for th in ths])
    ]

    roc_auc = skm.roc_auc_score(y_true, y_preds)
    average_precision = skm.average_precision_score(y_true, y_preds)
    max_f1 = skm.f1_score(y_true, y_preds>best_th)
    accuracy_on_max_f1 = skm.accuracy_score(y_true, y_preds>best_th)
    kappa = skm.cohen_kappa_score(y_true, y_preds>best_th)
    baseline=y_true.value_counts(True)

    return dict(
        roc_auc=roc_auc,
        average_precision=average_precision,
        max_f1=max_f1,
        accuracy_on_max_f1=accuracy_on_max_f1,
        kappa=kappa,
        baseline=baseline.iloc[0],
        best_th = best_th
    )

# Función mejorada para umbrales que da mayor ponderación a la sensibilidad
def get_medical_metrics(y_true, y_preds, sensitivity_weight=1.5):
    '''Calcula métricas con mayor peso a la sensibilidad (detección de verdaderos positivos)'''
    ths = np.linspace(0, 1, 1000)

    # Calcular precisión y recall para cada umbral
    scores = []
    for th in ths:
        y_pred = y_preds > th
        precision = skm.precision_score(y_true, y_pred, zero_division=0)
        recall = skm.recall_score(y_true, y_pred)

        # F-beta score con mayor peso al recall/sensibilidad
        fbeta = ((1 + sensitivity_weight**2) * precision * recall) /
                ((sensitivity_weight**2 * precision) + recall + 1e-10)
        scores.append(fbeta)

    best_idx = np.argmax(scores)
    best_th = ths[best_idx]

    # Cálculo de métricas estándar con el umbral optimizado
    y_pred = y_preds > best_th
    roc_auc = skm.roc_auc_score(y_true, y_preds)
    average_precision = skm.average_precision_score(y_true, y_preds)
    max_f1 = skm.f1_score(y_true, y_pred)
    accuracy = skm.accuracy_score(y_true, y_pred)
    kappa = skm.cohen_kappa_score(y_true, y_pred)
    sensitivity = skm.recall_score(y_true, y_pred)
    specificity = skm.recall_score(y_true, y_pred, pos_label=0)

    return dict(
        roc_auc=roc_auc,
        average_precision=average_precision,
        max_f1=max_f1,
        accuracy=accuracy,
        kappa=kappa,
        sensitivity=sensitivity,
        specificity=specificity,
        baseline=y_true.value_counts(True).iloc[0],
        best_th=best_th
    )

# 1. CARGA Y PREPARACIÓN DE DATOS
print("Cargando datos...")
df = pd.read_parquet("df_train.parquet")
df_test = pd.read_parquet("df_test.parquet")

# Calculamos la edad al momento de complicación
df['EDAD_COMPLICACION'] = (df['Fecha_cero'] - df['FECHA_NACIMIENTO']).dt.days // 365
df_test['EDAD_COMPLICACION'] = (df_test['Fecha_cero'] - df_test['FECHA_NACIMIENTO']).dt.days // 365

# División de datos
X = df.drop(columns=["Target"])
y = df["Target"]

# Verificamos la distribución de clases
class_distribution = y.value_counts(normalize=True) * 100
print(f"Distribución de clases: {class_distribution.iloc[1]:.2f}% positivos, {class_distribution.iloc[0]:.2f}% negativos")

# Porcentaje de nulidad
porcetaje_de_nulidad = (
    X.isnull()
    .apply(lambda s: s.value_counts(True)).T
)

porcetaje_de_nulidad.columns = ['not_null', 'null']
variables_muy_nulas = porcetaje_de_nulidad.query('null > 0.7').index

# Conversión de tipos de datos
columnas_numerico=['MULTI_CANCER','RIESGOS']
X[columnas_numerico] = X[columnas_numerico].astype(float)

columnas_categ= ['GENERO','ESTADO_CIVIL',
                 'CESION','CANCER_MAMA_FAMILIAR',
                'CANCER_OTRO_SITIO','CANCER_OTRO_SITIO_FAMILIAR','CEREBRAL_FAMILIAR',
                'atencion_nutricion'
                ]
X[columnas_categ] = X[columnas_categ].astype(str)

# Dividimos en entrenamiento y validación
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

# Validamos distribución en particiones
print("Distribución en conjunto de entrenamiento:")
print(y_train.value_counts(normalize=True) * 100)
print("Distribución en conjunto de validación:")
print(y_test.value_counts(normalize=True) * 100)

# 2. INGENIERÍA DE CARACTERÍSTICAS AVANZADA - CON MEJORAS
def create_medical_features(X):
    """Crea características específicas para complicaciones de cáncer de mama"""
    X_new = X.copy()

    # 1. Factor de riesgo compuesto (ponderado según importancia médica)
    if 'RIESGOS' in X.columns and 'CANCER_MAMA_FAMILIAR' in X.columns:
        X_new['RIESGO_COMPUESTO'] = X['RIESGOS'].astype(float) * (1 + 0.5*X['CANCER_MAMA_FAMILIAR'].astype(float))

        # Si existe la columna de cáncer en otro sitio, la incluimos
        if 'CANCER_OTRO_SITIO' in X.columns:
            X_new['RIESGO_COMPUESTO'] += 0.3*X['CANCER_OTRO_SITIO'].astype(float)

    # 2. Edad como factor de riesgo (relación no lineal con riesgo)
    if 'EDAD_COMPLICACION' in X.columns:
        # Relación cuadrática con la edad (mayor riesgo en edades extremas)
        X_new['EDAD_CUADRATICO'] = X['EDAD_COMPLICACION']**2 / 100

        # Categorización de edad (factor médicamente relevante)
        X_new['GRUPO_EDAD'] = pd.cut(
            X['EDAD_COMPLICACION'],
            bins=[0, 40, 50, 60, 70, 100],
            labels=['<40', '40-50', '50-60', '60-70', '>70']
        )

        # Interacción edad-riesgo
        if 'RIESGOS' in X.columns:
            X_new['EDAD_RIESGO'] = X['EDAD_COMPLICACION'] * X['RIESGOS'].astype(float) / 100

    # 3. Indicador de múltiples factores de riesgo
    risk_columns = ['CANCER_MAMA_FAMILIAR', 'CANCER_OTRO_SITIO', 'CANCER_OTRO_SITIO_FAMILIAR']
    risk_columns = [col for col in risk_columns if col in X.columns]

    if len(risk_columns) > 0:
        X_new['MULT_FACTORES_RIESGO'] = X[risk_columns].astype(float).sum(axis=1)

    # 4. Variables específicas para cáncer de mama
    if 'MULTI_CANCER' in X.columns:
        # Interacción con otros factores
        if 'RIESGOS' in X.columns:
            X_new['MULTI_RIESGOS'] = X['MULTI_CANCER'].astype(float) * X['RIESGOS'].astype(float)

    # 5. Interacción de variables categóricas con numéricas
    # Edad por género
    if 'EDAD_COMPLICACION' in X.columns and 'GENERO' in X.columns:
        for genero in X['GENERO'].unique():
            col_name = f"EDAD_GENERO_{genero}"
            X_new[col_name] = X['EDAD_COMPLICACION'] * (X['GENERO'] == genero).astype(float)

    # 6. Características específicas del dominio médico para cáncer de mama
    # Riesgo por edad categórica (importante médicamente)
    if 'RIESGOS' in X.columns and 'GRUPO_EDAD' in X_new.columns:
        for grupo in X_new['GRUPO_EDAD'].cat.categories:
            col_name = f"RIESGO_EDAD_{grupo}"
            X_new[col_name] = X['RIESGOS'].astype(float) * (X_new['GRUPO_EDAD'] == grupo).astype(float)

    # 7. Interacciones de factores de riesgo específicos
    if 'CANCER_MAMA_FAMILIAR' in X.columns and 'CANCER_OTRO_SITIO' in X.columns:
        X_new['RIESGO_FAMILIAR_COMBINADO'] = X['CANCER_MAMA_FAMILIAR'].astype(float) * X['CANCER_OTRO_SITIO'].astype(float) * 2

    # NUEVAS CARACTERÍSTICAS MEJORADAS

    # 8. Indicadores de alto riesgo para subgrupos específicos
    if 'RIESGOS' in X.columns and 'CANCER_MAMA_FAMILIAR' in X.columns:
        X_new['ALTO_RIESGO'] = ((X['RIESGOS'] > 2) &
                              (X['CANCER_MAMA_FAMILIAR'].astype(float) == 1)).astype(int)

    # 9. Características específicas para ancianos con cáncer múltiple (grupo de alto riesgo)
    if 'EDAD_COMPLICACION' in X.columns and 'MULTI_CANCER' in X.columns:
        X_new['RIESGO_EDAD_AVANZADA'] = ((X['EDAD_COMPLICACION'] > 65) &
                                      (X['MULTI_CANCER'] > 0)).astype(int)

    # 10. Interacciones no lineales basadas en conocimiento médico
    if 'EDAD_COMPLICACION' in X.columns and 'RIESGOS' in X.columns:
        # Riesgo exponencial en edades extremas (jóvenes y ancianos)
        edad_ref = X['EDAD_COMPLICACION'] - 55  # 55 como punto medio de referencia
        X_new['RIESGO_NO_LINEAL'] = np.exp(abs(edad_ref)/50) * X['RIESGOS'].astype(float)

    # 11. Factor de comorbilidad
    if 'MULTI_CANCER' in X.columns and 'RIESGOS' in X.columns and 'EDAD_COMPLICACION' in X.columns:
        X_new['COMORBILIDAD'] = X['MULTI_CANCER'] * (1 + X['EDAD_COMPLICACION']/70) * X['RIESGOS'].astype(float)

    return X_new

# Aplicamos la ingeniería de características
print("Aplicando ingeniería de características avanzada...")
X_train_enhanced = create_medical_features(X_train)
X_test_enhanced = create_medical_features(X_test)

# 3. PREPROCESAMIENTO
print("Configurando preprocesamiento...")

# Seleccionamos columnas categóricas y numéricas (excluyendo variables muy nulas)
categoricas = X_train_enhanced.select_dtypes(['object', 'category']).columns
categoricas = categoricas[~categoricas.isin(variables_muy_nulas)]

numericas = X_train_enhanced.select_dtypes(['int64', 'float64']).columns
numericas = numericas[~numericas.isin(variables_muy_nulas)]

# Transformadores para árbol (decisión, XGBoost)
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy='mean')),
    ("select_var", VarianceThreshold(0.1))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('dumm', OneHotEncoder(handle_unknown='ignore'))
])

# Preprocesador columnar para árboles
tree_preprocessing = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numericas),
    ("cat", categorical_transformer, categoricas)
])

# 4. EVALUACIÓN DE TÉCNICAS AVANZADAS DE SAMPLEO
print("Evaluando técnicas avanzadas de sampleo...")

# Definimos las técnicas de sampleo a evaluar
samplers = {
    'SMOTE': SMOTE(random_state=42, k_neighbors=7),  # Con k=7 según los resultados anteriores
    'BorderlineSMOTE': BorderlineSMOTE(random_state=42, k_neighbors=7),
    'ADASYN': ADASYN(random_state=42, n_neighbors=7),
    'SMOTEENN': SMOTEENN(random_state=42)
}

# Configuración base de XGBoost
config_xgb = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'use_label_encoder': False,
    'scale_pos_weight': 9,  # ~11% de casos positivos => ~9:1 ratio
    'random_state': 42,
    'tree_method': 'hist',
    'verbosity': 1,
    # Parámetros que funcionaron bien previamente
    'max_depth': 8,
    'learning_rate': 0.07,
    'n_estimators': 300,
    'subsample': 0.7,
    'colsample_bytree': 1.0,
    'min_child_weight': 1,
    'gamma': 0,
    'reg_alpha': 0.1,
    'reg_lambda': 1
}

# Comparamos las técnicas de sampleo
sampler_results = {}

# Solo evaluamos si no lo hemos hecho antes (para ahorrar tiempo)
if not os.path.exists("sampler_comparison.pkl"):
    for name, sampler in samplers.items():
        print(f"Evaluando {name}...")

        # Pipeline con el sampler específico
        pipeline = ImbPipeline([
            ('preprocesamiento', tree_preprocessing),
            ('sampling', sampler),
            ('model', XGBClassifier(**config_xgb))
        ])

        # Entrenamiento
        start = time()
        pipeline.fit(X_train_enhanced, y_train)
        train_time = time() - start

        # Predicciones
        y_pred_proba = pipeline.predict_proba(X_test_enhanced)[:, 1]

        # Métricas estándar y médicas (con mayor peso a sensibilidad)
        std_metrics = get_imbalaced_metrics(y_test, y_pred_proba)
        med_metrics = get_medical_metrics(y_test, y_pred_proba, sensitivity_weight=1.5)

        # Guardamos resultados
        sampler_results[name] = {
            'pipeline': pipeline,
            'std_metrics': std_metrics,
            'med_metrics': med_metrics,
            'train_time': train_time
        }

        print(f"{name} - F1: {std_metrics['max_f1']:.4f}, Sens: {med_metrics['sensitivity']:.4f}, Tiempo: {train_time:.2f}s")

    # Guardamos resultados para no tener que repetir
    with open("sampler_comparison.pkl", "wb") as f:
        pickle.dump(sampler_results, f)
else:
    # Cargamos resultados previos
    with open("sampler_comparison.pkl", "rb") as f:
        sampler_results = pickle.load(f)

    # Mostramos resultados
    for name, results in sampler_results.items():
        std_metrics = results['std_metrics']
        med_metrics = results['med_metrics']
        print(f"{name} - F1: {std_metrics['max_f1']:.4f}, Sens: {med_metrics['sensitivity']:.4f}")

# Seleccionamos el mejor sampler basado en F1 score
best_sampler_name = max(sampler_results.items(),
                      key=lambda x: x[1]['std_metrics']['max_f1'])[0]
best_sampler = samplers[best_sampler_name]
best_sampler_metrics = sampler_results[best_sampler_name]['std_metrics']

print(f"\nMejor técnica de sampleo: {best_sampler_name}")
print(f"F1-Score: {best_sampler_metrics['max_f1']:.4f}")
print(f"ROC AUC: {best_sampler_metrics['roc_auc']:.4f}")
print(f"Umbral óptimo: {best_sampler_metrics['best_th']:.4f}")

# 5. AJUSTE FINO DE HIPERPARÁMETROS CON EL MEJOR SAMPLER
print("\nRealizando ajuste fino de hiperparámetros con el mejor sampler...")

# Espacio de búsqueda refinado basado en los mejores parámetros encontrados anteriormente
param_dist = {
    'model__n_estimators': [200, 300, 400, 500],
    'model__max_depth': [7, 8, 9, 10],
    'model__learning_rate': [0.05, 0.07, 0.1, 0.12],
    'model__min_child_weight': [1, 2, 3],
    'model__subsample': [0.6, 0.7, 0.8],
    'model__colsample_bytree': [0.9, 1.0],
    'model__gamma': [0, 0.05, 0.1],
    'model__reg_alpha': [0.05, 0.1, 0.2],
    'model__reg_lambda': [0.8, 1, 1.2]
}

# Pipeline con el mejor sampler
xgb_pipeline = ImbPipeline([
    ('preprocesamiento', tree_preprocessing),
    ('sampling', best_sampler),
    ('model', XGBClassifier(**config_xgb))
])

# Validación cruzada estratificada con más repeticiones para mayor robustez
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Búsqueda aleatoria refinada
start = time()
random_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_dist,
    n_iter=20,  # Reducimos iteraciones para el ajuste fino
    scoring='f1',
    cv=cv,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train_enhanced, y_train)
end = time()

print(f"Tiempo de búsqueda: {timedelta(seconds=end-start)}")
print('Mejores hiperparámetros refinados:', random_search.best_params_)
print('Mejor F1-Score (CV):', random_search.best_score_)

# 6. EVALUACIÓN DEL MODELO OPTIMIZADO
print("\nEvaluando el modelo XGBoost optimizado...")

# Obtenemos el mejor modelo
best_model = random_search.best_estimator_

# Predicciones en validación
xgb_val_preds = best_model.predict_proba(X_test_enhanced)[:, 1]
xgb_metrics = get_imbalaced_metrics(y_test, xgb_val_preds)
xgb_medical_metrics = get_medical_metrics(y_test, xgb_val_preds, sensitivity_weight=1.5)

print("\nMétricas estándar:")
for key, value in xgb_metrics.items():
    print(f"{key}: {value}")

print("\nMétricas médicas (mayor peso a sensibilidad):")
for key, value in xgb_medical_metrics.items():
    print(f"{key}: {value}")

# 7. ANÁLISIS DE FALSOS NEGATIVOS PARA REFINAMIENTO
print("\nAnalizando falsos negativos para refinamiento...")

# Predicciones con umbral estándar
y_pred_std = (xgb_val_preds > xgb_metrics['best_th']).astype(int)
# Predicciones con umbral médico (más sensible)
y_pred_med = (xgb_val_preds > xgb_medical_metrics['best_th']).astype(int)

# Identificar falsos negativos con ambos umbrales
fn_std = (y_test == 1) & (y_pred_std == 0)
fn_med = (y_test == 1) & (y_pred_med == 0)

print(f"Falsos negativos con umbral estándar: {fn_std.sum()} de {(y_test == 1).sum()} positivos")
print(f"Falsos negativos con umbral médico: {fn_med.sum()} de {(y_test == 1).sum()} positivos")

# Comparamos características de falsos negativos vs. verdaderos positivos
fn_analysis = X_test_enhanced[fn_std].describe().T
tp_analysis = X_test_enhanced[(y_test == 1) & (y_pred_std == 1)].describe().T

# Solo columnas numéricas para análisis
num_cols = X_test_enhanced.select_dtypes(['int64', 'float64']).columns

# Calculamos diferencias porcentuales
comparison = pd.DataFrame({
    'Media en FN': fn_analysis.loc[num_cols, 'mean'],
    'Media en TP': tp_analysis.loc[num_cols, 'mean'],
})
comparison['% Diferencia'] = ((comparison['Media en FN'] / comparison['Media en TP']) - 1) * 100
comparison = comparison.sort_values('% Diferencia', key=abs, ascending=False)

print("\nCaracterísticas más diferenciadoras en falsos negativos:")
print(comparison.head(10))

# 8. ENTRENAMIENTO DEL MODELO FINAL CON PONDERACIÓN DE FALSOS NEGATIVOS
print("\nEntrenando modelo final con ponderación para reducir falsos negativos...")

# Combinar datos de entrenamiento y validación
X_full = pd.concat([X_train, X_test])
X_full_enhanced = create_medical_features(X_full)
y_full = pd.concat([y_train, y_test])

# Crear pesos para dar más importancia a falsos negativos históricos
sample_weights = np.ones(len(X_full))
sample_weights[y_full == 1] = 2  # Peso base para todos los positivos

# Entrenar modelo final con el mejor sampler y pesos
final_params = {k.replace('model__', ''): v
               for k, v in random_search.best_params_.items()
               if k.startswith('model__')}

# Pipeline final incluyendo las mejoras
final_pipeline = ImbPipeline([
    ('preprocesamiento', tree_preprocessing),
    ('sampling', best_sampler),
    ('model', XGBClassifier(**final_params))
])

# Entrenamiento con todos los datos y pesos
start = time()
final_pipeline.fit(X_full_enhanced, y_full)
end = time()
print(f"Tiempo de entrenamiento final: {timedelta(seconds=end-start)}")

# 9. EVALUACIÓN DE DIFERENTES UMBRALES DE DECISIÓN
print("\nEvaluando diferentes umbrales de decisión...")

# Mejor umbral basado en F1
best_f1_threshold = xgb_metrics['best_th']

# Mejor umbral basado en sensibilidad médica
best_medical_threshold = xgb_medical_metrics['best_th']

# Umbral medio entre F1 y médico (compromiso)
compromise_threshold = (best_f1_threshold + best_medical_threshold) / 2

# Evaluamos diferentes umbrales
thresholds = {
    'F1 Óptimo': best_f1_threshold,
    'Sensibilidad Médica': best_medical_threshold,
    'Compromiso': compromise_threshold
}

for name, threshold in thresholds.items():
    y_pred = (xgb_val_preds > threshold).astype(int)
    precision = skm.precision_score(y_test, y_pred)
    recall = skm.recall_score(y_test, y_pred)
    f1 = skm.f1_score(y_test, y_pred)
    acc = skm.accuracy_score(y_test, y_pred)

    print(f"\n{name} (umbral={threshold:.4f}):")
    print(f"Precision: {precision:.4f}")
    print(f"Recall/Sensibilidad: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print(f"Accuracy: {acc:.4f}")

# 10. GUARDAR MODELO FINAL Y DIFERENTES UMBRALES
print("\nGuardando modelo final y umbrales...")

with open("modelo_xgboost_final.pkl", "wb") as f:
    pickle.dump(final_pipeline, f)

# Guardamos todos los umbrales para uso futuro
thresholds_info = {
    'f1_optimal': best_f1_threshold,
    'medical_sensitivity': best_medical_threshold,
    'compromise': compromise_threshold
}

with open("umbrales_decision.pkl", "wb") as f:
    pickle.dump(thresholds_info, f)

# 11. GENERAR SUBMISSION FINAL CON EL UMBRAL SELECCIONADO
print("\nGenerando submission final...")

# Seleccionamos el umbral final a usar (puedes cambiar esto según necesidades)
# Usamos el umbral de compromiso para balance entre precisión y sensibilidad
final_threshold = compromise_threshold

# Preparamos datos de test
test_df = pd.read_parquet("df_test.parquet")
test_df['EDAD_COMPLICACION'] = (test_df['Fecha_cero'] - test_df['FECHA_NACIMIENTO']).dt.days // 365
test_df[columnas_numerico] = test_df[columnas_numerico].astype(float)
test_df[columnas_categ] = test_df[columnas_categ].astype(str)
test_df_enhanced = create_medical_features(test_df)

# Predicciones
submission_pred = final_pipeline.predict_proba(test_df_enhanced)[:, 1]
submission_pred_bool = submission_pred > final_threshold
submission_pred_int = [int(item) for item in submission_pred_bool]
submission = pd.DataFrame(data=dict(ID=test_df.index, Target=submission_pred_int))
submission.to_csv("submission_10.csv", index=False)

print("\nArchivo de submission generado: 'submission_10.csv'")
print(f"Umbral utilizado: {final_threshold:.4f} (compromiso entre F1 y sensibilidad médica)")

# Información adicional sobre distribución de predicciones
pred_distribution = pd.Series(submission_pred_int).value_counts(normalize=True) * 100
print(f"\nDistribución de predicciones en el conjunto de prueba:")
print(f"Clase 0 (Sin complicación): {pred_distribution.get(0, 0):.2f}%")
print(f"Clase 1 (Con complicación): {pred_distribution.get(1, 0):.2f}%")

SyntaxError: invalid syntax (<ipython-input-6-17b3ca2c0633>, line 71)

# Funciones Utiles

In [ ]:
# def get_imbalaced_metrics(y_true, y_preds):
#     '''calcula métricas de evaluación para modelos de clasificación cuando los datos están desbalanceados.'''
#     ths = np.linspace(0, 1, 1000)
#     best_th = ths[
#         np.argmax([skm.f1_score(y_true, y_preds>th) for th in ths])
#     ]

#     roc_auc = skm.roc_auc_score(y_true, y_preds)
#     average_precision = skm.average_precision_score(y_true, y_preds)
#     max_f1 = skm.f1_score(y_true, y_preds>best_th)
#     accuracy_on_max_f1 = skm.accuracy_score(y_true, y_preds>best_th)
#     kappa = skm.cohen_kappa_score(y_true, y_preds>best_th)
#     baseline=y_true.value_counts(True)


#     return dict(
#         roc_auc=roc_auc,
#         average_precision=average_precision,
#         max_f1=max_f1,
#         accuracy_on_max_f1=accuracy_on_max_f1,
#         kappa=kappa,
#         baseline=baseline.iloc[0],
#         best_th = best_th
#     )

# Carga de  Datos

In [ ]:
# df = pd.read_parquet("df_train.parquet")
# df.head()

# # Divicion de Dataset
# X, y = df.drop(columns="Target"), df["Target"]
# y.value_counts(True) * 100



,proportion
Target,
0.0,88.697851
1.0,11.302149


In [ ]:
# # Calculamos la edad de los pacientes al momento de la complicación o corte del analisis.
# X['EDAD_COMPLICACION'] = (X['Fecha_cero'] - X['FECHA_NACIMIENTO']).dt.days // 365

# #Porcentaje de Nulidad
# porcetaje_de_nulidad = (
#     X.isnull()
#     .apply(lambda s: s.value_counts(True)).T
# )

# porcetaje_de_nulidad.columns = ['not_null', 'null']
# variables_muy_nulas = porcetaje_de_nulidad.query('null > 0.7').index

In [ ]:
# #Convercion de tipos de Datos
# columnas_numerico=['MULTI_CANCER','RIESGOS']
# X[columnas_numerico] = X[columnas_numerico].astype(float)

# columnas_categ= ['GENERO','ESTADO_CIVIL',
#                  'CESION','CANCER_MAMA_FAMILIAR',
#                 'CANCER_OTRO_SITIO','CANCER_OTRO_SITIO_FAMILIAR','CEREBRAL_FAMILIAR'
#                 ,'atencion_nutricion'
#                 ]
# X[columnas_categ] = X[columnas_categ].astype(str)

In [ ]:
# #Dividimos el conjunto de datos en entrenamiento y prueba, por ahora, sin implementar un protocolo complejo de evaluación.
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

# #Validemos que tan desbalanceados quedaron los particionamientos
# print(y_train.value_counts(True)*100)
# print(y_test.value_counts(True)*100)


Target
0.0    88.693841
1.0    11.306159
Name: proportion, dtype: float64
Target
0.0    88.720539
1.0    11.279461
Name: proportion, dtype: float64


In [ ]:
# #Selecciona las columnas categóricas (variables tipo object o cadenas de texto) en X_train.
# categoricas = X_train.select_dtypes('object').columns
# categoricas = categoricas.delete(
#     categoricas.isin(variables_muy_nulas)
# )

# ##Selecciona las columnas numéricas en X_train (variables tipo int o float).
# numericas = X_train.select_dtypes('number').columns
# numericas = numericas.delete(
#     numericas.isin(variables_muy_nulas)
# )

In [ ]:
# #OneHotEncoder
# config_onehot = dict(
#     handle_unknown='ignore' # Ignora cualquier categoría desconocida que aparezca en los datos de prueba pero que no estaba en los datos de entrenamiento.
# )

# #
# numeric_transformer = Pipeline(
#     steps=[("imputer",  SimpleImputer(strategy='mean')),
#            ("scaler", StandardScaler()),
#            ("select_var", VarianceThreshold(0.1))
#            ]
# )

# categorical_transformer = Pipeline(
#     steps=[('imputer', SimpleImputer(strategy='most_frequent')),
#            ('dumm', OneHotEncoder(**config_onehot)),
#            ("selector", SelectPercentile(chi2, percentile=50))
#            ]
# )

# preprocessor = ColumnTransformer(
#     transformers=[
#         ("num", numeric_transformer, numericas),
#         ("cat", categorical_transformer, categoricas),
#     ]
# )

In [ ]:
# numeric_transformer = Pipeline(
#     steps=[("imputer",  SimpleImputer(strategy='mean')),
#            ("select_var", VarianceThreshold(0.1))
#            ]
# )

# categorical_transformer = Pipeline(
#     steps=[('imputer', SimpleImputer(strategy='most_frequent')),
#            ('dumm', OneHotEncoder(**config_onehot)),
#            ]
# )

# tree_preprocessing = ColumnTransformer(
#     transformers=[
#         ("num", numeric_transformer, numericas),
#         ("cat", categorical_transformer, categoricas),
#     ]
# )

# Modelo XGBOOST

In [ ]:
# config_xgb = {
#     'objective': 'binary:logistic',
#     'eval_metric': 'auc',
#     'use_label_encoder': False,
#     'scale_pos_weight': 9,  # ~11% de casos positivos => ~9:1 ratio
#     'random_state': 42,
#     'tree_method': 'hist',  # Más eficiente para grandes datasets
#     'verbosity': 1
# }
# param_dist = {
#     'model__n_estimators': [100, 200, 300, 500, 700, 1000],
#     'model__max_depth': [3, 4, 5, 6, 7, 8, 9],
#     'model__learning_rate': [0.01, 0.03, 0.05, 0.07, 0.1, 0.15],
#     'model__min_child_weight': [1, 3, 5, 7],
#     'model__subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
#     'model__colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
#     'model__gamma': [0, 0.1, 0.2, 0.3, 0.5],
#     'model__reg_alpha': [0, 0.001, 0.01, 0.1, 1],
#     'model__reg_lambda': [0, 0.001, 0.01, 0.1, 1],
#     'sampling__k_neighbors': [3, 5, 7],  # Optimizamos también los parámetros de SMOTE
# }


# xgb_pipeline = Pipeline([
#     ('preprocesamiento', tree_preprocessing),
#     ('classificador', XGBClassifier(**config_xgb))
# ])

In [ ]:
# start = time()
# xgb_pipeline.fit(X_train, y_train)
# end = time()

# print('Tiempo Entrenamiento XGBoost:', str(timedelta(seconds=end-start)))

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [19:54:28] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Tiempo Entrenamiento XGBoost: 0:00:00.426312


# Generar Submission

In [ ]:
# Entrenamiento con cronómetro
# start = time()
# xgb_pipeline.fit(X_train, y_train)
# end = time()

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [20:25:19] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


In [ ]:
# Tomar el mejor modelo
best_model = xgb_pipeline.best_estimator_

# Reentrenar con TODOS los datos (X e y completos)
best_model.fit(X, y)

NameError: name 'modelo_xgboost_final' is not defined

In [ ]:
# xgb_val_preds = best_model.predict_proba(X_test)[:, 1]
# xgb_metrics = get_imbalaced_metrics(y_test, xgb_val_preds)
# xgb_metrics

# with open("modelo_xgboost_final.pkl", "wb") as f:
#     pickle.dump(final_pipeline, f)

# test_df = pd.read_parquet( "df_test.parquet")
# test_df['EDAD_COMPLICACION'] = (test_df['Fecha_cero'] - test_df['FECHA_NACIMIENTO']).dt.days // 365

# test_df[columnas_numerico] = X[columnas_numerico].astype(float)
# test_df[columnas_categ] = X[columnas_categ].astype(str)

# submission_pred = best_model.predict_proba(test_df)[:, 1]
# submission_pred_bool = submission_pred>best_threshold_smote#best_th
# submission_pred_int = [int(item) for item in submission_pred_bool]
# submission = pd.DataFrame(data=dict(ID=test_df.index, Target=submission_pred_int))
# submission.to_csv("submission_8.csv", index=False)

In [ ]:
# with open("modelo_xgboost_final.pkl", "wb") as f:
#     pickle.dump(final_pipeline, f)

In [ ]:
# test_df = pd.read_parquet( "df_test.parquet")
# test_df['EDAD_COMPLICACION'] = (test_df['Fecha_cero'] - test_df['FECHA_NACIMIENTO']).dt.days // 365

# test_df[columnas_numerico] = X[columnas_numerico].astype(float)
# test_df[columnas_categ] = X[columnas_categ].astype(str)

# submission_pred = best_model.predict_proba(test_df)[:, 1]
# submission_pred_bool = submission_pred>best_threshold_smote#best_th
# submission_pred_int = [int(item) for item in submission_pred_bool]
# submission = pd.DataFrame(data=dict(ID=test_df.index, Target=submission_pred_int))
# submission.to_csv("submission_8.csv", index=False)

In [ ]:
submission.Target.value_counts()